# Paper result figures and tables — frozen experiments

Run from the project root. The notebook reads frozen results from Experiments 2, 3, 5, 6 and 7. It does not import any online client, regenerate objects, or alter the experiment caches.

**Only two output locations are written:** `paper_outputs/figures/*.pdf` and `paper_outputs/tables/*.csv`. PDF illustrations depict **predicted oriented bounding boxes**, not generated meshes. The selected 3D examples are annotated as representative (median-ranked within category) or diagnostic (worst relative outcome); diagnostic panels must not be presented as typical examples.

The optional parent-resize before/after figure is produced only if the local original raw caches are present and verified against the saved experimental measurements.

In [136]:
from pathlib import Path
import copy
import json
import math
import io
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

ROOT = Path.cwd().resolve()
FIG_DIR = ROOT / 'paper_outputs' / 'figures'
TABLE_DIR = ROOT / 'paper_outputs' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9.2,
    'axes.titlesize': 10.0,
    'axes.labelsize': 9.2,
    'xtick.labelsize': 8.4,
    'ytick.labelsize': 8.4,
    'legend.fontsize': 8.4,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.transparent': False,
})

METHODS = ('anchor', 'global', 'local')
DISPLAY = {'anchor': 'Anchor Tree', 'global': 'Global OBB', 'local': 'Hierarchy Local', 'random_anchor': 'Random Anchor'}
COLORS = {'anchor': '#286AA2', 'global': '#E18A37', 'local': '#389781', 'random_anchor': '#9A76B5'}
SCALES = (0.8, 1.2, 1.5)


def read_json(path):
    with Path(path).open('r', encoding='utf-8-sig') as stream:
        return json.load(stream)


def resolve(label, candidates, required=True):
    for rel in candidates:
        path = ROOT / rel
        if path.is_file():
            return path
    if required:
        raise FileNotFoundError(f'Missing {label}. Checked: {candidates}')
    return None


def save_fig(fig, outfile):
    filename = Path(str(outfile)).name

    # 每次重新从当前 notebook 工作目录构造路径
    output_dir = (
        Path.cwd()
        / "paper_outputs"
        / "figures"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = (
        output_dir
        / filename
    )

    # 删除可能存在的旧文件
    if output_path.exists():
        if output_path.is_dir():
            raise RuntimeError(
                f"Output path is unexpectedly a directory: "
                f"{output_path}"
            )

        try:
            output_path.unlink()
        except Exception as e:
            raise RuntimeError(
                "Cannot remove the existing PDF. "
                "Close the PDF if it is open in Acrobat, "
                "Edge, Chrome, etc.\n"
                f"Path: {output_path}\n"
                f"Error: {repr(e)}"
            )

    print("cwd:", repr(str(Path.cwd())))
    print("output_dir:", repr(str(output_dir)))
    print("output_path:", repr(str(output_path)))

    # 先测试 Windows 文件系统本身是否能写这个位置
    test_path = (
        output_dir
        / "__write_test__.tmp"
    )

    try:
        with test_path.open(
            "wb"
        ) as f:
            f.write(b"test")

        test_path.unlink()

    except Exception as e:
        raise RuntimeError(
            "Windows cannot write to the figure directory.\n"
            f"Directory: {output_dir}\n"
            f"Underlying error: {repr(e)}"
        )

    # 先让 Matplotlib 写到内存，避免 backend 直接处理 Windows path
    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.03
    )

    pdf_bytes = buffer.getvalue()

    buffer.close()

    # 再由 Python 自己写 PDF bytes
    try:
        with output_path.open(
            "wb"
        ) as f:
            f.write(pdf_bytes)

    except Exception as e:
        raise RuntimeError(
            "PDF rendering succeeded, but Windows failed "
            "when writing the PDF file.\n"
            f"Path: {repr(str(output_path))}\n"
            f"Error: {repr(e)}"
        )

    plt.close(fig)

    print(
        "Saved:",
        output_path
    )


def save_table(frame, filename):
    path = TABLE_DIR / filename
    frame.to_csv(path, index=False, encoding='utf-8-sig', float_format='%.9g')
    print('Table: ', path.relative_to(ROOT))


def pick_row(records, **conditions):
    for row in records:
        if all(row.get(k) == v for k, v in conditions.items()):
            return row
    raise KeyError(f'No result row for {conditions}')


def summary_mean(results, metric, method, group='summary', **filters):
    row = pick_row(results[group], metric=metric, method=method, **filters)
    return float(row['mean'])


def mean_by_method(records, method, metric, **filters):
    values = [float(row['metrics'][metric]) for row in records
              if row['method'] == method and all(row.get(k) == v for k, v in filters.items())
              and row.get('metrics', {}).get(metric) is not None]
    if not values:
        raise ValueError(f'No values for {method}, {metric}, {filters}')
    return float(np.mean(values))


def reduce_pct(anchor_value, baseline_value):
    return 100 * (baseline_value-anchor_value)/baseline_value if baseline_value > 0 else np.nan


def style_axis(ax, y_label=None):
    ax.grid(axis='y', alpha=0.15, linewidth=0.6)
    ax.set_axisbelow(True)
    if y_label:
        ax.set_ylabel(y_label)


def bar_values(ax, values, labels, title, colors, digits=3):
    x = np.arange(len(values))
    bars = ax.bar(x, values, width=0.62, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_xticks(x, labels, rotation=12, ha='right')
    ax.set_title(title, loc='left', fontweight='semibold')
    ax.set_ylim(0, max(values) * 1.23 if max(values) > 0 else 1)
    style_axis(ax)
    for b, v in zip(bars, values):
        ax.text(b.get_x()+b.get_width()/2, v, f'{v:.{digits}f}',
                ha='center', va='bottom', fontsize=8.1, color='#263440')

In [137]:
# Only frozen results are read. Optional detailed caches are used for verified edit visualizations.
P = {
 'partnet': resolve('PartNet corrected stats', [
     'partnet_results/experiments2/tables/results_tie_corrected.json',
     'partnet_results/tables/results_tie_corrected.json',
     'partnet_results/tables/results.json',
     'summary_src/stage2/tables/results_tie_corrected.json']),
 'same': resolve('same-geometry control', [
     'partnet_results/experiments2/same_geometry/results.json',
     'partnet_results/same_geometry/results.json',
     'summary_src/stage2/same_geometry/results.json']),
 'edit_summary': resolve('editing summary', [
     'partnet_results/experiments3/final_results.json',
     'experiments3_results/final_results.json',
     'summary_src/exp3/final_results.json']),
 'edit_tasks': resolve('editing tasks', [
     'partnet_results/experiments3/edit_tasks.json',
     'experiments3_results/edit_tasks.json',
     'summary_src/exp3/edit_tasks.json'], required=False),
 'edit_rows': resolve('editing object-level scores', [
     'partnet_results/experiments3/geometry_results.json',
     'experiments3_results/geometry_results.json',
     'summary_src/exp3/geometry_results.json'], required=False),
 'ow_eval': resolve('OpenWorld detailed evaluation', [
     'openworld_results/evaluation_all.json',
     'summary_src/exp5/evaluation_all.json']),
 'ow_pert': resolve('OpenWorld perturbation', [
     'openworld_results/perturbation_all.json',
     'summary_src/exp5/perturbation_all.json']),
 'ow_random': resolve('OpenWorld random-anchor ablation', [
     'openworld_results/random_anchor_all.json',
     'summary_src/exp5/random_anchor_all.json']),
 'ow_shared': resolve('OpenWorld shared structure', [
     'openworld_results/shared_specs.json',
     'summary_src/exp5/shared_specs.json']),
 'backbone': resolve('cross-backbone results', [
     'experiments6_results/qwen3_7_plus/results.json',
     'experiments6_results/results.json',
     'summary_src/exp6/results.json']),
 'qwen_partnet': resolve('PartNet Qwen object-level layouts', [
     'experiments6_results/qwen3_7_plus/partnet/evaluation_all.json',
     'experiments6_results/partnet/evaluation_all.json',
     'summary_src/exp6/partnet/evaluation_all.json']),
 'fidelity': resolve('text-structure fidelity', [
     'experiments7_results/results.json',
     'summary_src/exp7/results.json']),
 'primary_partnet_eval': resolve('PartNet original object-level layouts', [
     'partnet_results/experiments2/evaluation_all.json',
     'partnet_results/evaluation_all.json'], required=False),
 'primary_partnet_shared': resolve('PartNet shared specs', [
     'partnet_results/experiments2/shared_specs.json',
     'partnet_results/shared_specs.json'], required=False),
}
D = {key: read_json(path) if path else None for key, path in P.items()}
assert len(D['ow_eval']['complete_ids']) >= 100
assert len(D['qwen_partnet']['complete_ids']) >= 100
print('Frozen sources loaded:', ', '.join(k for k, v in D.items() if v is not None))

Frozen sources loaded: partnet, same, edit_summary, edit_tasks, edit_rows, ow_eval, ow_pert, ow_random, ow_shared, backbone, qwen_partnet, fidelity, primary_partnet_eval, primary_partnet_shared


In [174]:
# The 3D scenes are *oriented bounding-box proxies*, not reconstructed part meshes.
CORNERS = np.array([
    [-.5,-.5,-.5],[.5,-.5,-.5],[.5,.5,-.5],[-.5,.5,-.5],
    [-.5,-.5,.5],[.5,-.5,.5],[.5,.5,.5],[-.5,.5,.5]
], dtype=float)
FACES = ([0,3,2,1],[4,5,6,7],[0,1,5,4],[3,7,6,2],[0,4,7,3],[1,2,6,5])
LIGHT = np.array([-.35, -.55, .76], dtype=float)
LIGHT /= np.linalg.norm(LIGHT)
PART_PALETTE = list(plt.get_cmap('tab20').colors)


def corners(part):
    size = np.asarray(part['bbox_size'], dtype=float)
    rotation = np.asarray(part['world_rotation'], dtype=float)
    return (CORNERS * size) @ rotation.T + np.asarray(part['center'], dtype=float)


def bounds(layout):
    vertices = [corners(part) for part in layout if part.get('has_geometry', True)]
    if not vertices:
        raise ValueError('3D layout has no geometry-bearing parts')
    stack = np.concatenate(vertices, axis=0)
    return stack.min(axis=0), stack.max(axis=0)


def combine_bounds(layouts):
    boxes = [bounds(layout) for layout in layouts]
    return np.min([b[0] for b in boxes], axis=0), np.max([b[1] for b in boxes], axis=0)


def part_palette(records):

    paper_palette = [
        "#9677b8",
        "#e8d8fc",
        "#e31a1c",
        "#fb9a99",
        "#fdbf6f",
        "#b2df8a",
        "#33a02c"
    ]

    names = []

    for record in records:

        for part in record["layout_parts"]:

            name = part["name"]

            if name not in names:

                names.append(name)

    names = sorted(names)

    palette = {}

    for index, name in enumerate(names):

        palette[name] = paper_palette[index % len(paper_palette)]

    return palette

def obb_sat_margin(a, b):
    ra = np.asarray(a['world_rotation'], dtype=float)
    rb = np.asarray(b['world_rotation'], dtype=float)
    aa, bb = ra.T, rb.T
    axes = np.vstack((aa, bb, np.cross(aa[:, None, :], bb[None, :, :]).reshape(-1, 3)))
    magnitudes = np.linalg.norm(axes, axis=1)
    axes = axes[magnitudes > 1e-10] / magnitudes[magnitudes > 1e-10, None]
    delta = np.asarray(b['center'], dtype=float) - np.asarray(a['center'], dtype=float)
    r_a = np.abs(axes @ ra) @ (np.asarray(a['bbox_size'], dtype=float) / 2)
    r_b = np.abs(axes @ rb) @ (np.asarray(b['bbox_size'], dtype=float) / 2)
    return float(np.max(np.abs(axes @ delta) - r_a - r_b))


def shade_face(face, base_rgb, alpha=0.39):
    normal = np.cross(face[1] - face[0], face[2] - face[0])
    norm = np.linalg.norm(normal)
    n = normal / norm if norm > 1e-12 else np.array([0.,0.,1.])
    intensity = 0.66 + 0.31 * max(0., np.dot(n, LIGHT))
    rgb = np.clip(np.asarray(base_rgb[:3]) * intensity + (1-intensity)*np.ones(3), 0, 1)
    return (float(rgb[0]), float(rgb[1]), float(rgb[2]), alpha)


def render_boxes(ax, parts, palette, shared_bounds=None, connections=None, emphasize=None):

    edge_pairs = [
        (0, 1),
        (1, 2),
        (2, 3),
        (3, 0),
        (4, 5),
        (5, 6),
        (6, 7),
        (7, 4),
        (0, 4),
        (1, 5),
        (2, 6),
        (3, 7)
    ]

    face_indices = [
        (0, 1, 2, 3),
        (4, 5, 6, 7),
        (0, 1, 5, 4),
        (1, 2, 6, 5),
        (2, 3, 7, 6),
        (3, 0, 4, 7)
    ]

    part_by_id = {
        part["part_id"]: part
        for part in parts
    }

    all_points = []

    for part in parts:

        center = np.asarray(part["center"], dtype=float)

        size = np.asarray(part["bbox_size"], dtype=float)

        rotation = np.asarray(part["world_rotation"], dtype=float)

        hx = size[0] / 2

        hy = size[1] / 2

        hz = size[2] / 2

        local_corners = np.array([
            [-hx, -hy, -hz],
            [ hx, -hy, -hz],
            [ hx,  hy, -hz],
            [-hx,  hy, -hz],
            [-hx, -hy,  hz],
            [ hx, -hy,  hz],
            [ hx,  hy,  hz],
            [-hx,  hy,  hz]
        ])

        corners = local_corners @ rotation.T + center

        all_points.append(corners)

        part_color = palette.get(part["name"], "#9677b8")

        is_emphasized = emphasize is not None and part["part_id"] == emphasize

        face_alpha = 0.24 if not is_emphasized else 0.34

        box_linewidth = 0.8 if not is_emphasized else 1.8

        edge_color = "black"

        faces = [
            [corners[index] for index in face]
            for face in face_indices
        ]

        poly = Poly3DCollection(faces)

        poly.set_facecolor(part_color)

        poly.set_edgecolor(edge_color)

        poly.set_linewidth(box_linewidth)

        poly.set_alpha(face_alpha)

        ax.add_collection3d(poly)

        for start_index, end_index in edge_pairs:

            p1 = corners[start_index]

            p2 = corners[end_index]

            ax.plot(
                [p1[0], p2[0]],
                [p1[1], p2[1]],
                [p1[2], p2[2]],
                color=edge_color,
                linewidth=box_linewidth,
                alpha=0.90
            )

        if is_emphasized:

            ax.scatter(
                [center[0]],
                [center[1]],
                [center[2]],
                s=38,
                color="#fdbf6f",
                edgecolor="black",
                linewidth=0.8,
                depthshade=False,
                zorder=10
            )

    if connections is not None:

        for edge in connections:

            parent_id = edge["parent_part_id"]

            child_id = edge["child_part_id"]

            if parent_id not in part_by_id:

                continue

            if child_id not in part_by_id:

                continue

            parent = part_by_id[parent_id]

            child = part_by_id[child_id]

            parent_center = np.asarray(parent["center"], dtype=float)

            child_center = np.asarray(child["center"], dtype=float)

            margin = obb_sat_margin(parent, child)

            residual = abs(margin)

            if residual > 0.03:

                connection_color = "#e31a1c"

                connection_width = 2.1

                connection_alpha = 0.95

            else:

                connection_color = "#9677b8"

                connection_width = 1.5

                connection_alpha = 0.85

            ax.plot(
                [parent_center[0], child_center[0]],
                [parent_center[1], child_center[1]],
                [parent_center[2], child_center[2]],
                color=connection_color,
                linestyle="--",
                linewidth=connection_width,
                alpha=connection_alpha,
                zorder=8
            )

            ax.scatter(
                [parent_center[0]],
                [parent_center[1]],
                [parent_center[2]],
                s=18,
                color=connection_color,
                edgecolor="black",
                linewidth=0.5,
                depthshade=False,
                zorder=9
            )

            ax.scatter(
                [child_center[0]],
                [child_center[1]],
                [child_center[2]],
                s=18,
                color=connection_color,
                edgecolor="black",
                linewidth=0.5,
                depthshade=False,
                zorder=9
            )

    if shared_bounds is None:

        if len(all_points) == 0:

            mins = np.array([-0.5, -0.5, -0.5])

            maxs = np.array([0.5, 0.5, 0.5])

        else:

            stacked = np.concatenate(all_points, axis=0)

            mins = np.min(stacked, axis=0)

            maxs = np.max(stacked, axis=0)

    else:

        mins = np.asarray(shared_bounds[0], dtype=float)

        maxs = np.asarray(shared_bounds[1], dtype=float)

    center = (mins + maxs) / 2

    extent = maxs - mins

    span = float(np.max(extent))

    span = max(span, 1e-6)

    radius = span * 0.56

    ax.set_xlim(center[0] - radius, center[0] + radius)

    ax.set_ylim(center[1] - radius, center[1] + radius)

    ax.set_zlim(center[2] - radius, center[2] + radius)

    ax.set_box_aspect((1, 1, 1))

    ax.view_init(elev=20, azim=-55)

    ax.set_xticks([])

    ax.set_yticks([])

    ax.set_zticks([])

    ax.set_xlabel("")

    ax.set_ylabel("")

    ax.set_zlabel("")

    ax.grid(False)

    ax.set_axis_off()

    ax.set_facecolor("white")

    ax.xaxis.pane.set_alpha(0)

    ax.yaxis.pane.set_alpha(0)

    ax.zaxis.pane.set_alpha(0)


def group_triplets(records):
    grouped = {}
    for row in records:
        grouped.setdefault(str(row['object_id']), {})[row['method']] = row
    return {oid: rows for oid, rows in grouped.items() if all(m in rows for m in METHODS)}


def case_stats(oid, group):
    row = group['anchor']
    metric = 'shared_edge_obb_sat_deviation'
    anchor = float(group['anchor']['metrics'][metric])
    base = np.mean([float(group[m]['metrics'][metric]) for m in ('global','local')])
    return {'id': str(oid), 'category': row['category'],
            'nparts': len(row['layout_parts']), 'gain': float(base-anchor), 'rows': group}


def typical_cases(records, category_preferences, per_group=3, parts_min=4, parts_max=18):
    # Outcome-independent *within-category median rank* among valid positive examples.
    # The separate diagnostic gallery explicitly covers unfavorable outcomes.
    cases = [case_stats(oid, rows) for oid, rows in group_triplets(records).items()]
    cases = [c for c in cases if parts_min <= c['nparts'] <= parts_max]
    result = []
    for cat in category_preferences:
        within = sorted((c for c in cases if c['category']==cat), key=lambda c:c['gain'])
        positives = [c for c in within if c['gain'] > 1e-9]
        group = positives if positives else within
        if group:
            result.append(group[len(group)//2])
        if len(result) == per_group:
            break
    return result


def diagnostic_cases(partnet_records, openworld_records, partnet_label):
    output = []
    for dataset, records, desired in [('PartNet-'+partnet_label, partnet_records, 1),
                                      ('OpenWorld-DeepSeek', openworld_records, 2)]:
        cases = [case_stats(oid, rows) for oid, rows in group_triplets(records).items()]
        cases = [c for c in cases if 4 <= c['nparts'] <= 18 and c['gain'] < -1e-9]
        cases.sort(key=lambda c:c['gain'])
        for case in cases[:desired]:
            output.append({**case, 'dataset': dataset})
    return output


paper_palette = [
    "#9677b8",
    "#e8d8fc",
    "#e31a1c",
    "#fb9a99",
    "#fdbf6f",
    "#b2df8a",
    "#33a02c"
]

method_colors = {
    "anchor": "#9677b8",
    "global": "#e31a1c",
    "local": "#33a02c"
}

def build_triplet_palette(layouts):

    names = sorted({part["name"] for layout in layouts for part in layout})

    return {name: paper_palette[i % len(paper_palette)] for i, name in enumerate(names)}

def render_triplets(cases, outfile, header, show_edges=False, spec_by_id=None, diagnostic=False):

    if not cases:

        raise ValueError(f"No valid examples for {outfile}")

    rows = len(cases)

    fig = plt.figure(figsize=(12.2, 3.0 * rows + 0.55))

    gs = fig.add_gridspec(rows, 3, top=0.90, bottom=0.04, left=0.12, right=0.995, hspace=0.20, wspace=0.01)

    for i, case in enumerate(cases):

        layouts = [case["rows"][method]["layout_parts"] for method in METHODS]

        palette = build_triplet_palette(layouts)

        shared_bounds = combine_bounds(layouts)

        edges = None

        if show_edges:

            spec_row = spec_by_id.get(case["id"]) if spec_by_id else None

            if spec_row is None or spec_row.get("status") != "valid":

                raise ValueError(f'Shared specs unavailable for {case["id"]}')

            edges = spec_row["spec"]["connections"]

        for j, method in enumerate(METHODS):

            ax = fig.add_subplot(gs[i, j], projection="3d")

            render_boxes(ax, case["rows"][method]["layout_parts"], palette, shared_bounds=shared_bounds, connections=edges)

            if i == 0:

                ax.set_title(DISPLAY[method], fontsize=14, fontweight="semibold", pad=6, color=method_colors[method], loc="center")

            metrics = case["rows"][method]["metrics"]

            if show_edges:

                by_id = {part["part_id"]: part for part in case["rows"][method]["layout_parts"]}

                flagged = sum(abs(obb_sat_margin(by_id[edge["parent_part_id"]], by_id[edge["child_part_id"]])) > 0.03 for edge in edges)

                caption = f"Large-residual edges {flagged}/{len(edges)}   mean |m|={metrics['shared_edge_obb_sat_deviation']:.3f}"

            else:

                caption = f"SAT |m|={metrics['shared_edge_obb_sat_deviation']:.3f}   OBB ov.={metrics['obb_overlap_pair_fraction']:.2f}"

            ax.text2D(0.50, -0.035, caption, transform=ax.transAxes, ha="center", va="bottom", fontsize=11, color="black")

        label = f"{case['category']}\nID {case['id']}\n{case['nparts']} parts"

        if diagnostic:

            source = case["dataset"].replace("PartNet-", "PartNet / ").replace("OpenWorld-", "OpenWorld / ")

            label = source + "\n" + label + f"\nΔ={case['gain']:+.3f}"

        fig.text(0.028, 0.92 - (i + 0.5) * (0.86 / rows), label, va="center", ha="left", fontsize=12, color="black", fontweight="semibold")

    fig.suptitle(header, x=0.52, y=0.985, fontsize=14, fontweight="semibold")

    save_fig(fig, outfile)

# Prespecified categories; within category, choose middle-ranked positive case rather than extrema.
partnet_render_source = D['primary_partnet_eval'] if D['primary_partnet_eval'] is not None else D['qwen_partnet']
partnet_render_label = 'DeepSeek' if D['primary_partnet_eval'] is not None else 'Qwen3.7-Plus'
# 手动指定 Figure 2 中展示的 PartNet 对象
selected_partnet_ids = ["2472","22645","48847","2131",]
partnet_groups = group_triplets(partnet_render_source["objects"])

In [139]:
for i in range(100,120):
    print(list(partnet_groups.keys())[i])
    print(list(partnet_groups.values())[i]['anchor']['category'])

22234
Table
22359
Table
22534
Table
22604
Table
22645
Table
22895
Table
22961
Table
23149
Table
23640
Table
23673
Table
23805
Table
24034
Table
24189
Table
24218
Table
24243
Table
24341
Table
24355
Table
24398
Table
24460
Table
24862
Table


In [140]:
missing_ids = [oid for oid in selected_partnet_ids if oid not in partnet_groups]
if missing_ids:
    raise ValueError(f"These PartNet object IDs are unavailable in the current render source: {missing_ids}")
partnet_typical = [case_stats(oid,partnet_groups[oid]) for oid in selected_partnet_ids]
print("Selected PartNet objects:",[(case["category"],case["id"],case["nparts"],round(case["gain"], 4))for case in partnet_typical])
openworld_groups = group_triplets(D['ow_eval']['objects'])
rows = []
for oid, methods in openworld_groups.items():
    case = case_stats(oid,methods)
    rows.append({"object_id": oid,"category": case["category"],"part_count": case["nparts"],"anchor_gain": case["gain"],})
openworld_id_df = pd.DataFrame(rows)
display(openworld_id_df.sort_values(["category", "object_id"]).reset_index(drop=True))

Selected PartNet objects: [('Chair', '2472', 11, 0.0125), ('Table', '22645', 3, 0.0), ('StorageFurniture', '48847', 12, 0.0517), ('Faucet', '2131', 8, 0.0476)]


,object_id,category,part_count,anchor_gain
0,OW081,Appliances,7,1.013788e-01
1,OW082,Appliances,5,1.622635e-02
2,OW083,Appliances,5,7.694129e-03
3,OW084,Appliances,4,6.125787e-02
4,OW085,Appliances,5,5.204170e-18
...,...,...,...,...
114,OW036,Vehicles,7,1.973880e-02
115,OW037,Vehicles,13,6.944444e-03
116,OW038,Vehicles,10,4.398430e-04
117,OW039,Vehicles,11,1.856430e-02


In [141]:
selected_openworld_ids = ["OW081","OW059","OW036",]

In [142]:
openworld_groups = group_triplets(D['ow_eval']['objects'])
missing_ids = [oid for oid in selected_openworld_ids if oid not in openworld_groups]
if missing_ids:
    raise ValueError(f"Unavailable OpenWorld IDs: {missing_ids}")
openworld_typical = [case_stats(oid,openworld_groups[oid]) for oid in selected_openworld_ids]
negative_cases = diagnostic_cases(partnet_render_source['objects'], D['ow_eval']['objects'],partnet_render_label)
print('PartNet 3D cases:', [(c['category'],c['id']) for c in partnet_typical])
print('OpenWorld 3D cases:', [(c['category'],c['id']) for c in openworld_typical])
print('Diagnostics:', [(c['dataset'],c['id'],round(c['gain'],4)) for c in negative_cases])

PartNet 3D cases: [('Chair', '2472'), ('Table', '22645'), ('StorageFurniture', '48847'), ('Faucet', '2131')]
OpenWorld 3D cases: [('Appliances', 'OW081'), ('Tools', 'OW059'), ('Vehicles', 'OW036')]
Diagnostics: [('PartNet-DeepSeek', '48814', -0.1584), ('OpenWorld-DeepSeek', 'OW011', -0.069), ('OpenWorld-DeepSeek', 'OW063', -0.0495)]


In [143]:
# Seven insertion-ready result tables. Each row is grounded in frozen summary or object-level records.

# T1: GT-based PartNet and internal connection realization (including the adverse pose result).
t1 = pd.DataFrame([{
    'Method': DISPLAY[m],
    'Relative translation error (lower)': summary_mean(D['partnet'],'relative_translation_error',m),
    'World AABB size MAE (lower)': summary_mean(D['partnet'],'world_aabb_size_mae',m),
    'GT-edge SAT-margin MAE (lower)': summary_mean(D['partnet'],'gt_edge_bbox_margin_mae',m),
    'Shared-edge SAT deviation (lower)': summary_mean(D['partnet'],'shared_edge_obb_sat_deviation',m),
    'Predicted OBB overlap fraction (lower)': summary_mean(D['partnet'],'obb_overlap_pair_fraction',m),
} for m in METHODS])
save_table(t1,'table01_partnet_main.csv')

# T2: semantic-anchor ablation, separated by dataset.
ablation_rows = []
for dataset in ('PartNet','OpenWorld'):
    for method in ('anchor','random_anchor'):
        if dataset=='PartNet':
            dev = summary_mean(D['partnet'], 'shared_edge_obb_sat_deviation',method)
            overlap = summary_mean(D['partnet'],'obb_overlap_pair_fraction',method)
        elif method == 'anchor':
            dev = mean_by_method(D['ow_eval']['objects'], method, 'shared_edge_obb_sat_deviation')
            overlap = mean_by_method(D['ow_eval']['objects'], method, 'obb_overlap_pair_fraction')
        else:
            dev = mean_by_method(D['ow_random']['objects'], method, 'shared_edge_obb_sat_deviation')
            overlap = mean_by_method(D['ow_random']['objects'], method, 'obb_overlap_pair_fraction')
        ablation_rows.append({'Dataset':dataset, 'Method':DISPLAY[method],
                              'Shared-edge SAT deviation (lower)':dev,
                              'Predicted OBB overlap fraction (lower)':overlap})
save_table(pd.DataFrame(ablation_rows),'table02_semantic_anchor_ablation.csv')

# T3: child-size editing robustness; keep independent-generation and same-initial-geometry separate.
robust_rows = []
for regime in ('PartNet independent generation', 'PartNet same initial geometry', 'OpenWorld independent generation'):
    for scale in SCALES:
        for m in METHODS:
            if regime=='PartNet independent generation':
                value = summary_mean(D['partnet'],'margin_drift',m,group='perturbation_summary',scale=scale)
            elif regime=='PartNet same initial geometry':
                value = summary_mean(D['same'],'margin_drift',m,scale=scale)
            else:
                value = mean_by_method(D['ow_pert']['objects'],m,'margin_drift',scale=scale)
            robust_rows.append({'Regime':regime,'Child bbox scale':scale,'Method':DISPLAY[m],
                                'Target-edge SAT margin drift (lower)':value})
save_table(pd.DataFrame(robust_rows),'table03_size_perturbation.csv')

# T4: parent-size language-conditioned editing. Geometry plan is the frozen intended plan.
edit_rows = []
for regime in ('independent_generation', 'same_initial_geometry'):
    for m in METHODS:
        row = pick_row(D['edit_summary']['summary'],regime=regime,
                       subset='intended_plan_full',method=m,metric='outgoing_margin_drift')
        edit_rows.append({'Regime':regime.replace('_',' '),'Method':DISPLAY[m],
                          'Objects':row['n'], 'Outgoing margin drift (lower)':row['mean']})
save_table(pd.DataFrame(edit_rows),'table04_language_conditioned_editing.csv')

# T5: free-language structural layout; the volume is deliberately descriptive, not a quality ranking.
ow_rows = []
for m in METHODS:
    ow_rows.append({'Method':DISPLAY[m], 'Complete cases':len(D['ow_eval']['complete_ids']),
                    'Shared-edge SAT deviation (lower)':mean_by_method(D['ow_eval']['objects'],m,'shared_edge_obb_sat_deviation'),
                    'Predicted OBB overlap fraction (lower)':mean_by_method(D['ow_eval']['objects'],m,'obb_overlap_pair_fraction'),
                    'Part OBB volume sum (descriptive)':mean_by_method(D['ow_eval']['objects'],m,'part_volume_sum')})
save_table(pd.DataFrame(ow_rows),'table05_openworld_main.csv')

# T6: two-backbone common-cohort comparisons. Holm-adjusted p-values from the original analysis.
bb_rows = []
for row in D['backbone']['paired_statistics']:
    if row['metric'] not in ('shared_edge_obb_sat_deviation','margin_drift'):
        continue
    bb_rows.append({'Dataset':row['dataset'], 'Backbone':row['backbone'],
                    'Experiment':row['experiment'], 'Scale':row['scale'],
                    'Baseline':DISPLAY[row['baseline']], 'N':row['n'],
                    'Anchor mean':row['anchor_mean'], 'Baseline mean':row['baseline_mean'],
                    'Anchor advantage':row['mean_improvement'],
                    '95% CI low':row['ci_low'], '95% CI high':row['ci_high'],
                    'Holm-adjusted p':row['holm_p']})
save_table(pd.DataFrame(bb_rows),'table06_backbone_robustness.csv')

# T7: explicit text -> part graph fidelity. Strict lexical evaluation preserved.
fid_rows=[]
for model in ('deepseek','qwen3_7_plus'):
    fid_rows.append({'Backbone': 'DeepSeek' if model=='deepseek' else 'Qwen3.7-Plus',
        'Schema validity (%)': 100*pick_row(D['fidelity']['summary'],model=model,metric='schema_valid')['mean'],
        'Part instance F1 (%)':100*pick_row(D['fidelity']['summary'],model=model,metric='part_instance_f1')['mean'],
        'Connection F1 (%)':100*pick_row(D['fidelity']['summary'],model=model,metric='connection_f1')['mean'],
        'Root accuracy (%)':100*pick_row(D['fidelity']['summary'],model=model,metric='root_correct')['mean'],
        'Exact structure (%)':100*pick_row(D['fidelity']['summary'],model=model,metric='exact_structure')['mean']})
save_table(pd.DataFrame(fid_rows),'table07_explicit_structure_fidelity.csv')

Table:  paper_outputs\tables\table01_partnet_main.csv
Table:  paper_outputs\tables\table02_semantic_anchor_ablation.csv
Table:  paper_outputs\tables\table03_size_perturbation.csv
Table:  paper_outputs\tables\table04_language_conditioned_editing.csv
Table:  paper_outputs\tables\table05_openworld_main.csv
Table:  paper_outputs\tables\table06_backbone_robustness.csv
Table:  paper_outputs\tables\table07_explicit_structure_fidelity.csv


In [179]:
# Figure 1 — Main PartNet geometry/structure metrics

custom_palette = ["#9677b8", "#e8d8fc", "#e31a1c", "#fb9a99", "#fdbf6f", "#b2df8a", "#33a02c"]

method_colors = {METHODS[0]: custom_palette[0], METHODS[1]: custom_palette[2], METHODS[2]: custom_palette[6]}

method_labels = {METHODS[0]: "Anchor Tree", METHODS[1]: "Global\nOBB", METHODS[2]: "Hierarchy\nLocal"}

metric_specs = [("shared_edge_obb_sat_deviation", "Shared-edge SAT deviation ↓"), ("obb_overlap_pair_fraction", "Predicted OBB overlap ↓"), ("relative_translation_error", "Reference relative translation error ↓")]

fig, axs = plt.subplots(1, 3, figsize=(14.4, 4.6))

x = np.arange(len(METHODS))

for ax, (metric, title) in zip(axs, metric_specs):

    labels = [method_labels[m] for m in METHODS]

    values = [summary_mean(D["partnet"], metric, m) for m in METHODS]

    colors = [method_colors[m] for m in METHODS]

    bars = ax.bar(x, values, color=colors, width=0.62, edgecolor="black", linewidth=0.8)

    ax.set_xticks(x)

    ax.set_xticklabels(labels, fontsize=12)

    ax.set_title(title, fontsize=14, fontweight="semibold", pad=10, loc="center")

    ax.set_ylabel("Mean", fontsize=13)

    ax.tick_params(axis="x", labelsize=12, rotation=0, pad=5)

    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)

    ax.set_axisbelow(True)

    ymax = max(values) if max(values) > 0 else 1.0

    ax.set_ylim(0, ymax * 1.20)

    for bar, value in zip(bars, values):

        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + ymax * 0.025, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

fig.tight_layout(w_pad=2.6)

save_fig(fig, "fig01_partnet_main.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig01_partnet_main.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig01_partnet_main.pdf


In [175]:
# Figure 2 — Real Qwen-generated PartNet 3D box layouts, same view and part palette.
render_triplets(partnet_typical, "fig02_partnet_3d_comparison.pdf", f"Representative PartNet OBB Layouts | {partnet_render_label}")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig02_partnet_3d_comparison.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig02_partnet_3d_comparison.pdf


In [160]:
# Figure 3 — Semantic anchor ablation across PartNet and OpenWorld

semantic_anchor_color = "#9677b8"

random_anchor_color = "#e31a1c"

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.4))

metric_specs = [
    ("shared_edge_obb_sat_deviation", "Shared-edge SAT deviation ↓"),
    ("obb_overlap_pair_fraction", "Predicted OBB overlap fraction ↓")
]

for ax, (metric, title) in zip(axes, metric_specs):

    semantic_values = []

    random_values = []

    for domain in ["PartNet", "OpenWorld"]:

        if domain == "PartNet":

            semantic_values.append(summary_mean(D["partnet"], metric, "anchor"))

            random_values.append(summary_mean(D["partnet"], metric, "random_anchor"))

        else:

            semantic_values.append(mean_by_method(D["ow_eval"]["objects"], "anchor", metric))

            random_values.append(mean_by_method(D["ow_random"]["objects"], "random_anchor", metric))

    x = np.arange(2)

    width = 0.30

    bars_semantic = ax.bar(x - width / 2, semantic_values, width=width, color=semantic_anchor_color, edgecolor="black", linewidth=0.8, label="Semantic Anchor")

    bars_random = ax.bar(x + width / 2, random_values, width=width, color=random_anchor_color, edgecolor="black", linewidth=0.8, label="Random Anchor")

    ax.set_xticks(x)

    ax.set_xticklabels(["PartNet", "OpenWorld"], fontsize=12)

    ax.set_title(title, fontsize=14, fontweight="semibold", pad=10, loc="center")

    ax.set_ylabel("Mean", fontsize=13)

    ax.tick_params(axis="x", labelsize=12)

    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)

    ax.set_axisbelow(True)

    ymax = max(max(semantic_values), max(random_values))

    ax.set_ylim(0, ymax * 1.20)

    ax.legend(frameon=False, loc="upper right", fontsize=10)

    for bars in [bars_semantic, bars_random]:

        for bar in bars:

            value = bar.get_height()

            ax.text(bar.get_x() + bar.get_width() / 2, value + ymax * 0.025, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

fig.tight_layout(w_pad=2.2)

save_fig(fig, "fig03_semantic_anchor_ablation.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig03_semantic_anchor_ablation.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig03_semantic_anchor_ablation.pdf


In [182]:
# Figure 4 — Structural robustness under child-size perturbation

method_colors = {
    "anchor": "#9677b8",
    "global": "#e31a1c",
    "local": "#33a02c"
}

method_markers = {
    "anchor": "o",
    "global": "s",
    "local": "^"
}

annotation_offsets = {
    "anchor": 9,
    "global": 10,
    "local": -16
}

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.6), sharex=True)

settings = [
    ("PartNet / Independent", "partnet"),
    ("PartNet / Same Initial Geometry", "same"),
    ("OpenWorld / Independent", "openworld")
]

for ax, (title, kind) in zip(axes, settings):

    all_values = []

    for method in METHODS:

        if kind == "partnet":

            values = [summary_mean(D["partnet"], "margin_drift", method, group="perturbation_summary", scale=scale) for scale in SCALES]

        elif kind == "same":

            values = [summary_mean(D["same"], "margin_drift", method, scale=scale) for scale in SCALES]

        else:

            values = [mean_by_method(D["ow_pert"]["objects"], method, "margin_drift", scale=scale) for scale in SCALES]

        all_values.extend(values)

        ax.plot(SCALES, values, color=method_colors[method], marker=method_markers[method], linewidth=2.2, markersize=7.5, markeredgecolor="black", markeredgewidth=0.8, label=DISPLAY[method])

        for scale, value in zip(SCALES, values):

            offset = annotation_offsets[method]

            va = "bottom" if offset > 0 else "top"

            ax.annotate(f"{value:.3f}", xy=(scale, value), xytext=(0, offset), textcoords="offset points", ha="center", va=va, fontsize=11, color=method_colors[method])

    ax.set_xticks(SCALES)

    ax.set_xticklabels([str(scale) for scale in SCALES], fontsize=12)

    ax.set_xlabel("Child-size factor", fontsize=13)

    ax.set_title(title, fontsize=14, fontweight="semibold", pad=10, loc="center")

    ax.tick_params(axis="x", labelsize=12)

    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="both", linestyle="--", linewidth=0.6, alpha=0.35)

    ax.set_axisbelow(True)

    ymax = max(all_values)

    ax.set_ylim(0, ymax * 1.32)

    for spine in ax.spines.values():

        spine.set_edgecolor("black")

        spine.set_linewidth(0.8)

axes[0].set_ylabel("Shared-edge SAT margin drift ↓", fontsize=13)

axes[0].legend(frameon=False, loc="upper left", fontsize=10)

fig.tight_layout(w_pad=2.4)

save_fig(fig, "fig04_size_edit_robustness.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig04_size_edit_robustness.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig04_size_edit_robustness.pdf


In [164]:
# Figure 5 — Controlled language-conditioned parent editing

method_colors = {
    "anchor": "#9677b8",
    "global": "#e31a1c",
    "local": "#33a02c"
}

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.4), sharey=True)

regimes = [
    "independent_generation",
    "same_initial_geometry"
]

titles = [
    "Independent LLM Layouts",
    "Exactly Matched Initial Geometry"
]

for ax, regime, title in zip(axes, regimes, titles):

    values = [summary_mean(D["edit_summary"], "outgoing_margin_drift", method, regime=regime, subset="intended_plan_full") for method in METHODS]

    labels = [DISPLAY[method] for method in METHODS]

    colors = [method_colors[method] for method in METHODS]

    x = np.arange(len(METHODS))

    bars = ax.bar(x, values, width=0.62, color=colors, edgecolor="black", linewidth=0.8)

    ax.set_xticks(x)

    ax.set_xticklabels(labels, fontsize=12, rotation=0)

    ax.set_title(title, fontsize=14, fontweight="semibold", pad=10, loc="center")

    ax.tick_params(axis="x", labelsize=12)

    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)

    ax.set_axisbelow(True)

    ymax = max(values)

    ax.set_ylim(0, ymax * 1.22)

    for bar, value in zip(bars, values):

        ax.text(bar.get_x() + bar.get_width() / 2, value + ymax * 0.025, f"{value:.4f}", ha="center", va="bottom", fontsize=11)

axes[0].set_ylabel("Outgoing-edge SAT margin drift ↓", fontsize=13)

fig.tight_layout(w_pad=2.2)

save_fig(fig, "fig05_parent_editing.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig05_parent_editing.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig05_parent_editing.pdf


In [183]:
# Figure 6 — Independent free-language generation

method_colors = {
    "anchor": "#9677b8",
    "global": "#e31a1c",
    "local": "#33a02c"
}

metric_specs = [
    ("shared_edge_obb_sat_deviation", "Shared-edge SAT deviation ↓"),
    ("obb_overlap_pair_fraction", "Predicted OBB overlap ↓")
]

fig, axs = plt.subplots(1, 2, figsize=(10.6, 4.5))

x = np.arange(len(METHODS))

for ax, (metric, title) in zip(axs, metric_specs):

    values = [mean_by_method(D["ow_eval"]["objects"], method, metric) for method in METHODS]

    labels = [DISPLAY[method] for method in METHODS]

    colors = [method_colors[method] for method in METHODS]

    bars = ax.bar(x, values, width=0.62, color=colors, edgecolor="black", linewidth=0.8)

    ax.set_xticks(x)

    ax.set_xticklabels(labels, fontsize=12)

    ax.set_title(title, fontsize=14, fontweight="semibold", pad=10, loc="center")

    ax.set_ylabel("Mean", fontsize=13)

    ax.tick_params(axis="x", labelsize=12)

    ax.tick_params(axis="y", labelsize=11)

    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
    ax.set_axisbelow(True)
    ymax = max(values) if max(values) > 0 else 1.0
    ax.set_ylim(0, ymax * 1.22)

    for spine in ax.spines.values():

        spine.set_edgecolor("black")

        spine.set_linewidth(0.8)

    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + ymax * 0.025, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

fig.tight_layout(w_pad=2.4)

save_fig(fig, "fig06_openworld_main.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig06_openworld_main.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig06_openworld_main.pdf


In [176]:
# Figure 7 — Real DeepSeek-generated OpenWorld 3D box layouts; shared camera per object.
render_triplets(openworld_typical,'fig07_openworld_3d_comparison.pdf',
                'Free-language OBB layouts | DeepSeek | mid-ranked positive examples')

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig07_openworld_3d_comparison.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig07_openworld_3d_comparison.pdf


In [177]:
# Figure 8 — Honest limitations: worst relative SAT-deviation cases, including Anchor regressions.
render_triplets(negative_cases, "fig08_3d_diagnostic_failures.pdf", "Diagnostic 3D Layouts | Cases Where Anchor Trails the Baseline Mean", diagnostic=True)

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig08_3d_diagnostic_failures.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig08_3d_diagnostic_failures.pdf


In [152]:
ow_groups = group_triplets(D['ow_eval']['objects'])
rows = []
for oid, methods in ow_groups.items():
    case = case_stats(oid,methods)
    rows.append({"object_id": oid,"category": case["category"],"part_count": case["nparts"],"anchor_gain": case["gain"]})
ow_id_df = pd.DataFrame(rows)
display(ow_id_df[(ow_id_df["part_count"] >= 5)&(ow_id_df["part_count"] <= 10)&(ow_id_df["anchor_gain"] > 0)].sort_values("anchor_gain",ascending=False).head(20))

,object_id,category,part_count,anchor_gain
15,OW016,Furniture,10,0.293948
117,OW119,MechanicalComposite,7,0.171007
58,OW059,Tools,6,0.170647
56,OW057,Tools,5,0.138233
9,OW010,Furniture,5,0.113750
20,OW021,Vehicles,5,0.112991
45,OW046,Tools,5,0.110281
89,OW090,Appliances,5,0.101528
80,OW081,Appliances,7,0.101379
116,OW118,MechanicalComposite,6,0.091386


In [153]:
selected_edge_ids = ["OW090","OW077"]

In [178]:
ow_groups = group_triplets(D['ow_eval']['objects'])
missing_ids = [oid for oid in selected_edge_ids if oid not in ow_groups]
if missing_ids:
    raise ValueError(f"Unavailable OpenWorld IDs: {missing_ids}")
# Figure 9 — Same part layouts with the identical *inferred* shared tree edges overlaid.
# Red dashed edges indicate large SAT projection residual (threshold for display only).
shared_index={str(row['object_id']):row for row in D['ow_shared']['objects']}
edge_cases = [case_stats(oid,ow_groups[oid]) for oid in selected_edge_ids]
# Figure 9 — Shared structure edges visualized on OpenWorld 3D layouts.
render_triplets(edge_cases, "fig09_3d_shared_structure_edges.pdf", "Shared-Structure Edges on 3D Part Layouts | OpenWorld DeepSeek", show_edges=True, spec_by_id=shared_index)

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig09_3d_shared_structure_edges.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig09_3d_shared_structure_edges.pdf


In [180]:
# Figure 10 — Effect-size reduction under two representation-generation backbones

from matplotlib.colors import LinearSegmentedColormap

dataset_display = {"partnet": "PartNet", "openworld": "OpenWorld"}

backbone_display = {"primary_deepseek": "DeepSeek", "qwen3.7-plus": "Qwen3.7+"}

baseline_display = {"global": "Global OBB", "local": "Hierarchy Local"}

labels = []

values = []

for dataset in ("partnet", "openworld"):

    for backbone in ("primary_deepseek", "qwen3.7-plus"):

        for baseline in ("global", "local"):

            labels.append(f"{dataset_display[dataset]} | {backbone_display[backbone]} | vs {baseline_display[baseline]}")

            row_values = []

            static = pick_row(D["backbone"]["paired_statistics"], dataset=dataset, backbone=backbone, baseline=baseline, experiment="static", metric="shared_edge_obb_sat_deviation", scale=None)

            row_values.append(reduce_pct(static["anchor_mean"], static["baseline_mean"]))

            for scale in SCALES:

                row = pick_row(D["backbone"]["paired_statistics"], dataset=dataset, backbone=backbone, baseline=baseline, experiment="scale_perturbation", metric="margin_drift", scale=scale)

                row_values.append(reduce_pct(row["anchor_mean"], row["baseline_mean"]))

            values.append(row_values)

arr = np.asarray(values)

paper_cmap = LinearSegmentedColormap.from_list("paper_purple", ["#ffffff", "#e8d8fc", "#9677b8"])

fig, ax = plt.subplots(figsize=(10.6, 5.6))

im = ax.imshow(arr, cmap=paper_cmap, aspect="auto", vmin=0, vmax=100)

ax.set_yticks(np.arange(len(labels)))

ax.set_yticklabels(labels, fontsize=11)

ax.set_xticks(np.arange(4))

ax.set_xticklabels(["Static SAT deviation", "Scale 0.8", "Scale 1.2", "Scale 1.5"], fontsize=12)

ax.set_title("Anchor advantage under shared structure and two LLM backbones", fontsize=14, fontweight="semibold", pad=10, loc="center")

ax.tick_params(axis="x", labelsize=12)

ax.tick_params(axis="y", labelsize=11)

for spine in ax.spines.values():

    spine.set_edgecolor("black")

    spine.set_linewidth(0.8)

for i in range(arr.shape[0]):

    for j in range(arr.shape[1]):

        value = arr[i, j]

        text_color = "white" if value >= 55 else "black"

        ax.text(j, i, f"{value:.0f}%", ha="center", va="center", fontsize=11, color=text_color)

cbar = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.024)

cbar.set_label("Reduction vs baseline (%)", fontsize=13)

cbar.ax.tick_params(labelsize=11)

cbar.outline.set_edgecolor("black")

cbar.outline.set_linewidth(0.8)

fig.tight_layout()

save_fig(fig, "fig10_backbone_generalization.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig10_backbone_generalization.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig10_backbone_generalization.pdf


In [181]:
# Figure 11 — Explicit text-to-structure fidelity

from matplotlib.colors import LinearSegmentedColormap

models = ("deepseek", "qwen3_7_plus")

metrics = (("part_instance_f1", "Part F1"), ("connection_f1", "Connection F1"), ("exact_structure", "Exact structure"))

model_display = {"deepseek": "DeepSeek", "qwen3_7_plus": "Qwen3.7+"}

model_colors = {"deepseek": "#9677b8", "qwen3_7_plus": "#e31a1c"}

category_display = {"MechanicalComposite": "Mechanical Composite"}

paper_cmap = LinearSegmentedColormap.from_list("paper_purple", ["#ffffff", "#e8d8fc", "#9677b8"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.6, 4.8), gridspec_kw={"width_ratios": [1.0, 1.35]})

x = np.arange(3)

width = 0.35

for i, model in enumerate(models):

    scores = [100 * pick_row(D["fidelity"]["summary"], model=model, metric=key)["mean"] for key, _ in metrics]

    offset = (-0.5 if i == 0 else 0.5) * width

    bars = ax1.bar(x + offset, scores, width, label=model_display[model], color=model_colors[model], edgecolor="black", linewidth=0.8)

    for bar, value in zip(bars, scores):

        ax1.text(bar.get_x() + bar.get_width() / 2, value + 0.6, f"{value:.1f}", ha="center", va="bottom", fontsize=11)

ax1.set_xticks(x)

ax1.set_xticklabels([title for _, title in metrics], fontsize=12)

ax1.set_ylim(0, 106)

ax1.set_ylabel("Score (%)", fontsize=13)

ax1.set_title("Explicit-constraint recovery", fontsize=14, fontweight="semibold", pad=10, loc="center")

ax1.tick_params(axis="x", labelsize=12)

ax1.tick_params(axis="y", labelsize=11)

ax1.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)

ax1.set_axisbelow(True)

ax1.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=11)

for spine in ax1.spines.values():

    spine.set_edgecolor("black")

    spine.set_linewidth(0.8)

cats = sorted({row["category"] for row in D["fidelity"]["category_summary"]})

cat_labels = [category_display.get(cat, cat) for cat in cats]

values = np.array([[100 * pick_row(D["fidelity"]["category_summary"], model=model, category=cat, metric="exact_structure")["mean"] for cat in cats] for model in models])

im = ax2.imshow(values, cmap=paper_cmap, vmin=90, vmax=100, aspect="auto")

ax2.set_yticks(np.arange(len(models)))

ax2.set_yticklabels([model_display[model] for model in models], fontsize=11)

ax2.set_xticks(np.arange(len(cats)))

ax2.set_xticklabels(cat_labels, rotation=28, ha="right", fontsize=12)

ax2.set_title("Strict exact structure by category (%)", fontsize=14, fontweight="semibold", pad=10, loc="center")

ax2.tick_params(axis="x", labelsize=12)

ax2.tick_params(axis="y", labelsize=11)

for spine in ax2.spines.values():

    spine.set_edgecolor("black")

    spine.set_linewidth(0.8)

for i in range(values.shape[0]):

    for j in range(values.shape[1]):

        value = values[i, j]

        text_color = "white" if value >= 96 else "black"

        ax2.text(j, i, f"{value:.0f}", ha="center", va="center", fontsize=11, color=text_color)

cbar = fig.colorbar(im, ax=ax2, pad=0.016, fraction=0.039)

cbar.set_label("Exact structure (%)", fontsize=13)

cbar.ax.tick_params(labelsize=11)

cbar.outline.set_edgecolor("black")

cbar.outline.set_linewidth(0.8)

fig.tight_layout(rect=(0, 0.06, 1, 1), w_pad=2.0)

save_fig(fig, "fig11_text_structure_fidelity.pdf")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig11_text_structure_fidelity.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig11_text_structure_fidelity.pdf


In [170]:
# Figure 12 — VERIFIED real language edit: before/after native parameter updates

paper_palette = [
    "#9677b8",
    "#e8d8fc",
    "#e31a1c",
    "#fb9a99",
    "#fdbf6f",
    "#b2df8a",
    "#33a02c"
]

method_colors = {
    "anchor": "#9677b8",
    "global": "#e31a1c",
    "local": "#33a02c"
}


def quaternion_to_matrix_wxyz(q):

    q = np.asarray(q, dtype=float)

    q = q / np.linalg.norm(q)

    w, x, y, z = q

    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def layout_from_raw(raw, spec, method, normal):

    parameter = {part["part_id"]: part for part in raw["parts"]}

    meta = {part["part_id"]: part for part in spec["parts"]}

    parent_of = {edge["child_part_id"]: edge["parent_part_id"] for edge in spec["connections"]}

    memo = {}

    def solve(pid):

        if pid in memo:

            return memo[pid]

        raw_part = parameter[pid]

        size = np.asarray(raw_part["bbox_size"], dtype=float)

        if method == "global" or pid == spec["root_part_id"]:

            center = np.asarray(raw_part["center"], dtype=float)

            rotation = quaternion_to_matrix_wxyz(raw_part["rotation"])

        else:

            parent = solve(parent_of[pid])

            parent_center = np.asarray(parent["center"])

            parent_rotation = np.asarray(parent["world_rotation"])

            parent_size = np.asarray(parent["bbox_size"])

            if method == "local":

                rotation = parent_rotation @ quaternion_to_matrix_wxyz(raw_part["relative_rotation"])

                center = parent_center + parent_rotation @ np.asarray(raw_part["relative_position"], dtype=float)

            else:

                rotation = parent_rotation @ quaternion_to_matrix_wxyz(raw_part["rotation"])

                parent_anchor = np.asarray(raw_part["parent_anchor"], dtype=float)

                child_anchor = np.asarray(raw_part["child_anchor"], dtype=float)

                offset = np.asarray(raw_part["offset"], dtype=float)

                center = parent_center + parent_rotation @ (parent_size * parent_anchor + offset) - rotation @ (size * child_anchor)

        item = {**meta[pid], "center": center.tolist(), "bbox_size": size.tolist(), "world_rotation": rotation.tolist(), "has_geometry": True}

        memo[pid] = item

        return item

    world = [solve(part["part_id"]) for part in spec["parts"]]

    shift = np.asarray(normal["center"], dtype=float)

    scale = float(normal["scale"])

    if not np.isfinite(scale) or scale <= 0:

        raise ValueError("Bad original normalization")

    for part in world:

        part["center"] = ((np.asarray(part["center"]) - shift) / scale).tolist()

        part["bbox_size"] = (np.asarray(part["bbox_size"]) / scale).tolist()

    return world


def raw_edit(raw, task):

    output = copy.deepcopy(raw)

    plan = task["expected_plan"]

    axis = {"x": 0, "y": 1, "z": 2}[plan["axis"]]

    selected = [part for part in output["parts"] if part["part_id"] == plan["target_part_id"]]

    if len(selected) != 1:

        raise ValueError("Selected editing target absent/duplicated")

    selected[0]["bbox_size"][axis] *= float(plan["scale_factor"])

    return output


def layout_max_error(a, b):

    ad = {row["part_id"]: row for row in a}

    bd = {row["part_id"]: row for row in b}

    if ad.keys() != bd.keys():

        raise ValueError("Before layout part inventory mismatch")

    return max(float(np.max(np.abs(np.asarray(ad[pid][key]) - np.asarray(bd[pid][key])))) for pid in ad for key in ("center", "bbox_size", "world_rotation"))


def outgoing_drift(before, after, spec, plan):

    target = plan["target_part_id"]

    outgoing = [edge for edge in spec["connections"] if edge["parent_part_id"] == target]

    before_by_id = {part["part_id"]: part for part in before}

    after_by_id = {part["part_id"]: part for part in after}

    if not outgoing:

        raise ValueError("Edit target has no child")

    return float(np.mean([abs(obb_sat_margin(before_by_id[target], before_by_id[edge["child_part_id"]]) - obb_sat_margin(after_by_id[target], after_by_id[edge["child_part_id"]])) for edge in outgoing]))


def prepare_verified_edit_example():

    if not all((D["edit_tasks"], D["edit_rows"], D["primary_partnet_eval"], D["primary_partnet_shared"])):

        return None

    root = P["primary_partnet_eval"].parent

    shared_items = D["primary_partnet_shared"]["objects"] if isinstance(D["primary_partnet_shared"], dict) else D["primary_partnet_shared"]

    shared = {str(item["object_id"]): item for item in shared_items}

    original = {(str(row["object_id"]), row["method"]): row for row in D["primary_partnet_eval"]["objects"]}

    metrics = {(str(row["object_id"]), row["method"]): row for row in D["edit_rows"]["records"] if row["regime"] == "independent_generation"}

    candidates = []

    for task in D["edit_tasks"]["tasks"]:

        oid = str(task["object_id"])

        if not all((oid, method) in metrics for method in METHODS):

            continue

        if task["outgoing_edge_count"] < 1:

            continue

        part_count = len(shared.get(oid, {}).get("spec", {}).get("parts", []))

        if not (4 <= part_count <= 18):

            continue

        score = float(np.mean([metrics[oid, method]["metrics"]["outgoing_margin_drift"] for method in ("global", "local")]) - metrics[oid, "anchor"]["metrics"]["outgoing_margin_drift"])

        moved = metrics[oid, "anchor"]["metrics"]["moved_descendant_count"]

        if score > 0.001 and moved > 0:

            candidates.append((score, task))

    candidates.sort(key=lambda row: row[0], reverse=True)

    for _, task in candidates:

        oid = str(task["object_id"])

        paths = {method: root / "generation" / method / (oid + ".json") for method in METHODS}

        if not all(path.is_file() for path in paths.values()):

            continue

        spec = shared[oid]["spec"]

        scenes = {}

        for method in METHODS:

            cached = read_json(paths[method])

            if cached.get("status") != "valid":

                raise RuntimeError(f"Former valid object cache is now invalid: {paths[method]}")

            raw = cached["result"]

            saved = original[oid, method]

            if not saved.get("layout_parts"):

                raise RuntimeError("Frozen primary geometry is absent")

            before = layout_from_raw(raw, spec, method, saved["normalization"])

            error = layout_max_error(before, saved["layout_parts"])

            if error > 1e-8:

                raise ValueError(f"Original geometry mismatch {oid}:{method}: {error}")

            after = layout_from_raw(raw_edit(raw, task), spec, method, saved["normalization"])

            drift = outgoing_drift(before, after, spec, task["expected_plan"])

            reported = float(metrics[oid, method]["metrics"]["outgoing_margin_drift"])

            if abs(drift - reported) > 1e-8:

                raise ValueError(f"Edited geometry metric mismatch {oid}:{method}: {drift} != {reported}")

            scenes[method] = {"before": before, "after": after, "drift": drift}

        return {"task": task, "scenes": scenes, "spec": spec}

    return None


verified_edit = prepare_verified_edit_example()

if verified_edit is not None:

    scene = verified_edit["scenes"]

    all_layouts = [scene[method][phase] for method in METHODS for phase in ("before", "after")]

    view_box = combine_bounds(all_layouts)

    part_names = sorted({part["name"] for layout in all_layouts for part in layout})

    palette = {name: paper_palette[index % len(paper_palette)] for index, name in enumerate(part_names)}

    target = verified_edit["task"]["expected_plan"]["target_part_id"]

    fig = plt.figure(figsize=(10.4, 9.8))

    gs = fig.add_gridspec(3, 2, top=0.90, bottom=0.08, left=0.14, right=0.985, hspace=0.14, wspace=0.025)

    for i, method in enumerate(METHODS):

        for j, phase in enumerate(("before", "after")):

            ax = fig.add_subplot(gs[i, j], projection="3d")

            render_boxes(ax, scene[method][phase], palette, shared_bounds=view_box, emphasize=target)

            if i == 0:

                column_title = "Original Layout" if j == 0 else "After Controlled Parent Resize"

                ax.set_title(column_title, fontsize=14, fontweight="semibold", pad=2, loc="center")

            if j == 0:

                fig.text(0.025, 0.80 - i * 0.27, DISPLAY[method], ha="left", va="center", fontsize=13, color=method_colors[method], fontweight="semibold")

            if j == 1:

                ax.text2D(0.5, -0.04, f"Outgoing SAT drift = {scene[method]['drift']:.4f}", transform=ax.transAxes, ha="center", fontsize=11, color="black")

    #instruction = verified_edit["task"]["instruction"]

    figure_title = f"Controlled Language Edit | {verified_edit['task']['category']} {verified_edit['task']['object_id']}"

    fig.suptitle(figure_title, y=0.985, fontsize=14, fontweight="semibold")

    #fig.text(0.14, 0.03, instruction, fontsize=11, color="black")

    save_fig(fig, "fig12_verified_parent_edit_3d.pdf")

else:

    stale_optional = FIG_DIR / "fig12_verified_parent_edit_3d.pdf"

    if stale_optional.is_file():

        stale_optional.unlink()

    print("Optional 3D edit figure not produced: original raw generation caches not available.")

cwd: 'D:\\MyJupyter\\Works\\ASPT'
output_dir: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures'
output_path: 'D:\\MyJupyter\\Works\\ASPT\\paper_outputs\\figures\\fig12_verified_parent_edit_3d.pdf'
Saved: D:\MyJupyter\Works\ASPT\paper_outputs\figures\fig12_verified_parent_edit_3d.pdf


In [158]:
# Final verification: precisely named paper figures and tables, all graphics PDF.
expected_pdf = {
 'fig01_partnet_main.pdf', 'fig02_partnet_3d_comparison.pdf',
 'fig03_semantic_anchor_ablation.pdf', 'fig04_size_edit_robustness.pdf',
 'fig05_parent_editing.pdf', 'fig06_openworld_main.pdf',
 'fig07_openworld_3d_comparison.pdf', 'fig08_3d_diagnostic_failures.pdf',
 'fig09_3d_shared_structure_edges.pdf', 'fig10_backbone_generalization.pdf',
 'fig11_text_structure_fidelity.pdf',
}
if verified_edit is not None:
    expected_pdf.add('fig12_verified_parent_edit_3d.pdf')
expected_csv = {
 'table01_partnet_main.csv','table02_semantic_anchor_ablation.csv',
 'table03_size_perturbation.csv','table04_language_conditioned_editing.csv',
 'table05_openworld_main.csv','table06_backbone_robustness.csv',
 'table07_explicit_structure_fidelity.csv',
}
for name in expected_pdf:
    path = FIG_DIR / name
    assert path.is_file() and path.stat().st_size > 600, f'Missing PDF {name}'
    assert path.read_bytes()[:4] == b'%PDF', f'Not a real PDF: {name}'
for name in expected_csv:
    path = TABLE_DIR / name
    assert path.is_file() and path.stat().st_size > 80, f'Missing table {name}'
print(f'Paper outputs completed: {len(expected_pdf)} PDF figures and {len(expected_csv)} CSV tables.')
print('Optional verified editing render:', 'included' if verified_edit else 'not available')

Paper outputs completed: 12 PDF figures and 7 CSV tables.
Optional verified editing render: included
